Check problems with taxonomy missing. Get also the Genus taxonomy 

TODO give the option to not save with header and index to the check_save_file

In [1]:
import sys
sys.path.append('../../')

from file_management import get_files_dir,check_save_file
_, INPUT_DIR, OUTPUT_DIR = get_files_dir()

import pandas as pd
from Bio import Entrez

Entrez.email = 'elisa.m.zavala@ntnu.no'

In [2]:
INPUT_DIR

'/Users/elisamarquez/Documents/PhD/2semestre/Engineering_db/Engineering_db/files/Input'

In [3]:
# Load the NCBI Taxonomy database into a Pandas DataFrame.

full_tax_file = INPUT_DIR+'/Taxonomy/ncbi_lineages_2022-11-19.csv'
full_tax  =  pd.read_csv(full_tax_file,dtype=str)
full_tax.shape

(2468763, 69)

In [4]:
# Lets see data 
full_tax.loc[full_tax.tax_id=='1143'].dropna(axis=1)

,tax_id,superkingdom,phylum,order,family,genus,species,clade,clade1,no rank,no rank1
879,1143,Bacteria,Cyanobacteria,Synechococcales,Merismopediaceae,Synechocystis,Synechocystis sp.,Terrabacteria group,Cyanobacteria/Melainabacteria group,cellular organisms,unclassified Synechocystis


In [5]:
# Remove undesired taxonomic levels.
# The levels to remove are defined in the dictionary 'levels_remove'.
# The keys in this dictionary correspond to the taxonomic levels,
# while the values are lists of the names of the taxonomic groups to remove
# from the DataFrame.

levels_remove = {'kingdom':['Metazoa','Viridiplantae'],
                 'superkingdom':['Viruses'],
                 'no rank':['other entries','environmental samples'], # Remove things like Vectors 
                 'no rank1':["environmental samples"],# Do not remove all unclassified  (Synechocystis sp.)
                'no rank2':["metagenomes", "environmental samples",
                            'miscellaneous sequences','mixed libraries'],# Do not remove all unclassified from rank 2 for Genus
                "no rank3":["environmental samples"]} 

for level, remove_tax in levels_remove.items():
    neg_bool = full_tax.loc[:,level].isin(remove_tax) 
    full_tax = full_tax.loc[~neg_bool]

In [6]:
# Cases to take into account
full_tax.loc[full_tax.tax_id.isin(['106590','53482']),['tax_id','species','no rank1','no rank2','strain']]


,tax_id,species,no rank1,no rank2,strain
32681,53482,Cupriavidus necator,NaN,NaN,Cupriavidus necator H850
78817,106590,Cupriavidus necator,NaN,NaN,NaN


In [7]:
full_tax.loc[full_tax.genus.isin(['713630']),:].dropna(axis=1)


,tax_id,superkingdom,phylum,class,order,family,genus,species,biotype,clade,...,subphylum,subsection,subspecies,subtribe,superclass,superfamily,superorder,superphylum,tribe,varietas


In [9]:
# Select only organisms at the species level that do not have a substrain.
species = full_tax.loc[:,['tax_id','genus','species','no rank1','no rank2','no rank3','strain']].dropna(subset=['species'])
species = species[species.strain.isnull()]
species = species.loc[:, species.columns != 'strain']

# Drop duplicate entries and keep the first occurrence.
species = species.drop_duplicates(subset=['species'], keep='first')

In [10]:
# Seems all of them would give us plant names so delete them
species.loc[species.genus.isin(['Candidatus Phytoplasma']),:]


,tax_id,genus,species,no rank1,no rank2,no rank3
1695,2155,Candidatus Phytoplasma,Candidatus Phytoplasma sp.,unclassified phytoplasmas,NaN,NaN
15275,33016,Candidatus Phytoplasma,Group III phytoplasma,NaN,NaN,NaN
15959,33927,Candidatus Phytoplasma,Oenothera phytoplasma,unclassified phytoplasmas,NaN,NaN
15960,33928,Candidatus Phytoplasma,Faba bean phyllody phytoplasma,NaN,NaN,NaN
17401,35770,Candidatus Phytoplasma,Tomato big bud phytoplasma,NaN,NaN,NaN
...,...,...,...,...,...,...
2468500,2994032,Candidatus Phytoplasma,'Prunus persica' yellowing and reddening of le...,NaN,NaN,NaN
2468501,2994033,Candidatus Phytoplasma,'Datura stramonium' little leaf phytoplasma,NaN,NaN,NaN
2468502,2994034,Candidatus Phytoplasma,'Catharanthus roseus' little leaf and yellowin...,NaN,NaN,NaN
2468522,2994079,Candidatus Phytoplasma,'Cassia fistula' flat stem phytoplasma,NaN,NaN,NaN


In [11]:
# Remove undesired taxonomic groups from the DataFrame.
#Phytoplasma
species = species.loc[~species.genus.isin(['Candidatus Phytoplasma']),:]


## Symbiont 
species = species.loc[~species.species.str.lower().str.contains('symbiont'),:]
species = species.loc[~species.species.str.lower().str.contains('symbiotic'),:]
species = species.loc[~species.species.str.lower().str.contains('endophyte'),:]
species = species.loc[~species.species.str.lower().str.contains('cyanobiont'),:]
species = species.loc[~species.species.str.lower().str.contains('endosybmiont'),:]
species = species.loc[~species.species.str.lower().str.contains('endosybiont'),:]
species = species.loc[~species.species.str.lower().str.contains('endobymbiont'),:]

## Parasites 

species = species.loc[~species.species.str.lower().str.contains('parasite'),:]

## Contamination
species = species.loc[~species.species.str.lower().str.contains('contaminant'),:]

## Manure pit (swine), rumen or activated sludges
species = species.loc[~species.species.str.lower().str.contains('swine'),:]
species = species.loc[~species.species.str.lower().str.contains('rumen'),:]
species = species.loc[~species.species.str.lower().str.contains('activated sludge'),:]

 
# Mixed cultures
species = species.loc[~species.species.str.lower().str.contains('mixed culture'),:]

# Remove 'Candidatus' from the species names
species.species = species.species.str.split('Candidatus ').str[-1]

In [12]:
# Overview
new_tax_table = species.copy()
new_tax_table.tail(5)

,tax_id,genus,species,no rank1,no rank2,no rank3
2468751,2995172,Pseudomonas,Pseudomonas sp. S1Bt3,unclassified Pseudomonas,NaN,NaN
2468752,2995173,Pseudomonas,Pseudomonas sp. S1Bt7,unclassified Pseudomonas,NaN,NaN
2468753,2995174,Pseudomonas,Pseudomonas sp. S1Bt42,unclassified Pseudomonas,NaN,NaN
2468756,2995225,Sphingobacterium,Sphingobacterium sp. UT-1RO-CII-1,unclassified Sphingobacterium,NaN,NaN
2468758,2995235,Scleroderma,Scleroderma venenatum (nom. inval.),unclassified Scleroderma,NaN,NaN


In [13]:
# Separate genus and species into two separate columns
new_tax_table.loc[:,'Genus'] = new_tax_table.species.str.split(' ').str[0]
new_tax_table.Genus = new_tax_table.Genus.str.strip("'").str.strip("[").str.strip("]")

# Remove lowercase genus names
new_tax_table = new_tax_table.loc[~new_tax_table.Genus.str.islower()]

# Remove species that contain 'Candidatus' and 'of' in the same name
new_tax_table = new_tax_table.loc[~((new_tax_table.species.str.contains(' of ') )& (new_tax_table.Genus == 'Candidatus')) , :]

In [14]:
genus_check = new_tax_table.Genus.value_counts()
genus_check.head(10)

Bacillus          30491
Pseudomonas       27730
Streptomyces      22635
Vibrio            13424
Bradyrhizobium     8997
Burkholderia       6689
Arthrobacter       5805
Enterobacter       5456
Fusarium           5065
Rhizobium          5046
Name: Genus, dtype: int64

In [15]:
# Combine the first letter of the genus name and the species name, e.g. E. coli

new_tax_table.loc[:,'Species'] = new_tax_table.species.str.split(' ',1).str[1]
new_tax_table.loc[:,'Species'] = new_tax_table.Genus.str[0] +'. '+new_tax_table.Species

In [16]:
# Check duplicated values
print(new_tax_table.shape)
species_check = new_tax_table.species.value_counts()
species_check.head(5)

(582332, 8)


Elioraea thermophila        2
Nitrosopumilus sp. DDS1     2
Omnitrophica bacterium      2
Nitrosopumilus sp. SW       2
Azorhizobium caulinodans    1
Name: species, dtype: int64

In [17]:
# Remove duplicate species names
new_tax_table = new_tax_table.drop_duplicates(subset=['species'], keep='first')
new_tax_table.shape

(582328, 8)

In [18]:
# Format data 
new_tax_table.rename(columns={'species':'Full_name'},inplace=True)
new_tax_table.Full_name = new_tax_table.Full_name.str.strip("'").str.strip("(").str.strip(")")
new_tax_table.Full_name = new_tax_table.Full_name.str.strip("[").str.strip("]")
new_tax_table = new_tax_table.loc[:,['tax_id','Full_name','Genus','Species']]

In [19]:
new_tax_table.head()

,tax_id,Full_name,Genus,Species
3,7,Azorhizobium caulinodans,Azorhizobium,A. caulinodans
4,9,Buchnera aphidicola,Buchnera,B. aphidicola
6,11,Cellulomonas gilvus,Cellulomonas,C. gilvus
8,14,Dictyoglomus thermophilum,Dictyoglomus,D. thermophilum
10,17,Methylophilus methylotrophus,Methylophilus,M. methylotrophus


In [24]:
new_tax_table.loc[new_tax_table.Genus=='Nicotiana']

,tax_id,Full_name,Genus,Species


In [21]:
check_save_file(new_tax_table,'Filtered_taxonomy.csv','Taxonomy',input_dir=True)

Saved file in: /Users/elisamarquez/Documents/PhD/2semestre/Engineering_db/Engineering_db/files/Input/Taxonomy/Filtered_taxonomy.csv
